Ingeniería de características

1. ¿Cual es nuestro objetivo?

Recordemos que queremos construir:

Un modelo de Machine Learning capaz de predecir si el equipo local ganará un partido de fútbol americano antes de que comience.

Nuestra variable objetivo será:

homeWin
- 1: gana el equipo local.
- 0: el equipo local no gana.

In [5]:
import pandas as pd
import numpy as np

# Cargar dataset limpio
df = pd.read_csv("../data/partidos_2024_limpio.csv")

# Mostrar dimensiones
print("Dimensiones:", df.shape)

# Visualizar registros
df.head()

Dimensiones: (3801, 34)


,id,season,week,seasonType,startDate,startTimeTBD,completed,neutralSite,conferenceGame,attendance,...,awayConference,awayPoints,awayLineScores,awayPostgameWinProbability,awayPregameElo,awayPostgameElo,excitementIndex,highlights,notes,playoff
0,401693677,2024,1,regular,2024-08-24T04:00:00.000Z,True,True,True,False,NaN,...,NaN,45.0,"[14, 7, 14, 10]",NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,401635525,2024,1,regular,2024-08-24T16:00:00.000Z,False,True,True,True,47998.0,...,ACC,21.0,"[8, 6, 0, 7]",0.065470,1692.0,1676.0,4.952562,NaN,Aer Lingus College Football Classic,NaN
2,401654665,2024,1,regular,2024-08-24T19:30:00.000Z,False,True,False,False,16125.0,...,Southland,23.0,"[7, 0, 3, 13]",0.261285,NaN,NaN,4.170973,NaN,NaN,NaN
3,401643697,2024,1,regular,2024-08-24T20:00:00.000Z,False,True,False,False,17314.0,...,Big Sky,35.0,"[0, 14, 0, 21]",0.440510,NaN,NaN,5.319335,NaN,NaN,NaN
4,401654662,2024,1,regular,2024-08-24T23:00:00.000Z,False,True,True,False,4358.0,...,UAC,15.0,"[8, 7, 0, 0]",0.020046,NaN,NaN,5.330265,NaN,FCS Kickoff,NaN


Paso 1. Inspeccionar las variables del dataset

In [10]:
# Crear una copia de trabajo
df_fe = df.copy()

# Conservar partidos completados con resultados disponibles
df_fe = df_fe[
    (df_fe["completed"] == True) &
    (df_fe["homePoints"].notna()) &
    (df_fe["awayPoints"].notna())
].copy()

# Identificar partidos empatados
empates = df_fe["homePoints"] == df_fe["awayPoints"]

print("Partidos empatados:", empates.sum())

# Excluir empates para trabajar con una clasificación binaria
df_fe = df_fe.loc[~empates].copy()

# Crear variable objetivo
# 1 = ganó el equipo local
# 0 = ganó el equipo visitante
df_fe["homeWin"] = (
    df_fe["homePoints"] > df_fe["awayPoints"]
).astype(int)

# Mostrar resultados
print("\nDimensiones finales:", df_fe.shape)

print("\nDistribución de homeWin:")
print(df_fe["homeWin"].value_counts())

print("\nPorcentajes:")
print(df_fe["homeWin"].value_counts(normalize=True) * 100)

Partidos empatados: 13

Dimensiones finales: (3786, 35)

Distribución de homeWin:
homeWin
1    2182
0    1604
Name: count, dtype: int64

Porcentajes:
homeWin
1    57.633386
0    42.366614
Name: proportion, dtype: float64


Recuerda nuestro objetivo: predecir el ganador antes de que comience el partido.

Por tanto, no podemos utilizar información que solamente conocemos después del encuentro.

Revisar las variables que podemos conservar

Con este análisis identificaremos:

- Qué variables tienen demasiados valores faltantes.
- Cuáles son numéricas y cuáles categóricas.
- Qué información podemos utilizar antes del partido.
- Qué variables pueden aportar información sobre la fortaleza de cada equipo.

In [11]:
# Variables candidatas para predecir el resultado
columnas_candidatas = [
    "season",
    "week",
    "seasonType",
    "startDate",
    "startTimeTBD",
    "neutralSite",
    "conferenceGame",
    "venueId",
    "venue",
    "homeId",
    "homeTeam",
    "homeClassification",
    "homeConference",
    "homePregameElo",
    "awayId",
    "awayTeam",
    "awayClassification",
    "awayConference",
    "awayPregameElo"
]

# Revisar disponibilidad de información
resumen = pd.DataFrame({
    "tipo_dato": df_fe[columnas_candidatas].dtypes.astype(str),
    "valores_faltantes": df_fe[columnas_candidatas].isnull().sum(),
    "porcentaje_faltante": (
        df_fe[columnas_candidatas].isnull().mean() * 100
    ).round(2),
    "valores_unicos": df_fe[columnas_candidatas].nunique()
})

resumen.sort_values(
    by="porcentaje_faltante",
    ascending=False
)

,tipo_dato,valores_faltantes,porcentaje_faltante,valores_unicos
awayPregameElo,float64,2988,78.92,521
homePregameElo,float64,2867,75.73,550
awayConference,str,52,1.37,73
awayClassification,str,52,1.37,4
homeConference,str,15,0.40,73
homeClassification,str,15,0.40,4
venue,str,8,0.21,700
venueId,float64,8,0.21,705
seasonType,str,0,0.00,2
season,int64,0,0.00,1


nuestro primer conjunto de características

In [12]:
# Seleccionar características iniciales
features = [
    "week",
    "seasonType",
    "neutralSite",
    "conferenceGame",
    "homeId",
    "awayId",
    "homeClassification",
    "awayClassification",
    "homeConference",
    "awayConference",
    "startTimeTBD"
]

# Crear dataset de características
df_features = df_fe[features + ["homeWin"]].copy()

# Crear características temporales
df_features["startDate"] = pd.to_datetime(
    df_fe["startDate"],
    errors="coerce"
)

df_features["month"] = df_features["startDate"].dt.month
df_features["dayOfWeek"] = df_features["startDate"].dt.dayofweek

# Eliminar fecha original
df_features.drop(columns=["startDate"], inplace=True)

# Revisar resultado
print("Dimensiones:", df_features.shape)

print("\nCaracterísticas creadas:")
print(df_features.columns.tolist())

print("\nPrimeros registros:")
display(df_features.head())

Dimensiones: (3786, 14)

Características creadas:
['week', 'seasonType', 'neutralSite', 'conferenceGame', 'homeId', 'awayId', 'homeClassification', 'awayClassification', 'homeConference', 'awayConference', 'startTimeTBD', 'homeWin', 'month', 'dayOfWeek']

Primeros registros:


C:\Users\maxwe\AppData\Local\Temp\ipykernel_49260\1639657160.py:26: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df_features["dayOfWeek"] = df_features["startDate"].dt.dayofweek
C:\Users\maxwe\AppData\Local\Temp\ipykernel_49260\1639657160.py:29: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df_features.drop(columns=["startDate"], inplace=True)


,week,seasonType,neutralSite,conferenceGame,homeId,awayId,homeClassification,awayClassification,homeConference,awayConference,startTimeTBD,homeWin,month,dayOfWeek
0,1,regular,True,False,124179,108382,ii,NaN,Independent DII,NaN,True,0,8,5
1,1,regular,True,True,59,52,fbs,fbs,ACC,ACC,False,1,8,5
2,1,regular,False,False,2627,2377,fcs,fcs,UAC,Southland,False,1,8,5
3,1,regular,False,False,167,147,fbs,fcs,Mountain West,Big Sky,False,0,8,5
4,1,regular,True,False,2546,2453,fcs,fcs,Big South-OVC,UAC,False,1,8,5


¿Qué acabamos de hacer?

Creamos dos nuevas características:
- month: mes en el que se disputa el partido.
- dayOfWeek: día de la semana (lunes = 0, domingo = 6).
También conservamos homeWin, que será nuestra variable objetivo.

Importante: los identificadores homeId y awayId se tratarán como variables categóricas, no como cantidades numéricas.

Auditoría del dataset de características

In [13]:
# 1. Dimensiones
print("DIMENSIONES:", df_features.shape)

# 2. Valores faltantes
print("\nVALORES FALTANTES:")
print(
    df_features.isnull()
    .sum()
    .sort_values(ascending=False)
)

# 3. Tipos de datos
print("\nTIPOS DE DATOS:")
print(df_features.dtypes)

# 4. Distribución de la variable objetivo
print("\nDISTRIBUCIÓN DE homeWin:")
print(df_features["homeWin"].value_counts())

# 5. Verificar duplicados
print("\nREGISTROS DUPLICADOS:")
print(df_features.duplicated().sum())

# 6. Verificar que no existan columnas de resultados finales
print("\nCOLUMNAS DEL DATASET:")
print(df_features.columns.tolist())

DIMENSIONES: (3786, 14)

VALORES FALTANTES:
awayConference        52
awayClassification    52
homeConference        15
homeClassification    15
week                   0
seasonType             0
awayId                 0
homeId                 0
conferenceGame         0
neutralSite            0
startTimeTBD           0
homeWin                0
month                  0
dayOfWeek              0
dtype: int64

TIPOS DE DATOS:
week                  int64
seasonType              str
neutralSite            bool
conferenceGame         bool
homeId                int64
awayId                int64
homeClassification      str
awayClassification      str
homeConference          str
awayConference          str
startTimeTBD           bool
homeWin               int64
month                 int32
dayOfWeek             int32
dtype: object

DISTRIBUCIÓN DE homeWin:
homeWin
1    2182
0    1604
Name: count, dtype: int64

REGISTROS DUPLICADOS:
0

COLUMNAS DEL DATASET:
['week', 'seasonType', 'neutralSite', 'con

In [14]:
print("DIMENSIONES:", df_features.shape)

print("\nVALORES FALTANTES:")
print(df_features.isnull().sum())

print("\nDUPLICADOS:", df_features.duplicated().sum())

print("\nDISTRIBUCIÓN HOMEWIN:")
print(df_features["homeWin"].value_counts())

DIMENSIONES: (3786, 14)

VALORES FALTANTES:
week                   0
seasonType             0
neutralSite            0
conferenceGame         0
homeId                 0
awayId                 0
homeClassification    15
awayClassification    52
homeConference        15
awayConference        52
startTimeTBD           0
homeWin                0
month                  0
dayOfWeek              0
dtype: int64

DUPLICADOS: 0

DISTRIBUCIÓN HOMEWIN:
homeWin
1    2182
0    1604
Name: count, dtype: int64


In [15]:
# Identificar variables categóricas
columnas_categoricas = [
    "seasonType",
    "homeClassification",
    "awayClassification",
    "homeConference",
    "awayConference"
]

# Rellenar valores faltantes
df_features[columnas_categoricas] = (
    df_features[columnas_categoricas]
    .fillna("Unknown")
)

# Comprobar resultado
print("VALORES FALTANTES DESPUÉS DEL TRATAMIENTO:")
print(df_features.isnull().sum())

print("\nDIMENSIONES:")
print(df_features.shape)

VALORES FALTANTES DESPUÉS DEL TRATAMIENTO:
week                  0
seasonType            0
neutralSite           0
conferenceGame        0
homeId                0
awayId                0
homeClassification    0
awayClassification    0
homeConference        0
awayConference        0
startTimeTBD          0
homeWin               0
month                 0
dayOfWeek             0
dtype: int64

DIMENSIONES:
(3786, 14)


Revisar la cardinalidad

In [16]:
# Identificar variables categóricas
columnas_categoricas = [
    "seasonType",
    "homeClassification",
    "awayClassification",
    "homeConference",
    "awayConference",
    "homeId",
    "awayId"
]

# Cantidad de categorías únicas
print("CATEGORÍAS POR VARIABLE:\n")

for columna in columnas_categoricas:
    print(f"{columna}: {df_features[columna].nunique()} categorías")

CATEGORÍAS POR VARIABLE:

seasonType: 2 categorías
homeClassification: 5 categorías
awayClassification: 5 categorías
homeConference: 74 categorías
awayConference: 74 categorías
homeId: 679 categorías
awayId: 700 categorías


Separar entrenamiento y prueba

In [17]:
# Identificar las semanas disponibles
semanas = sorted(df_features["week"].unique())

print("Semanas disponibles:", semanas)

# Utilizar las primeras semanas para entrenamiento
# y las últimas para prueba
semana_corte = int(len(semanas) * 0.80)

semanas_train = semanas[:semana_corte]
semanas_test = semanas[semana_corte:]

# Separar los conjuntos
train = df_features[
    df_features["week"].isin(semanas_train)
].copy()

test = df_features[
    df_features["week"].isin(semanas_test)
].copy()

# Separar características y variable objetivo
X_train = train.drop(columns=["homeWin"])
y_train = train["homeWin"]

X_test = test.drop(columns=["homeWin"])
y_test = test["homeWin"]

# Mostrar resultados
print("\nSemanas de entrenamiento:", semanas_train)
print("Semanas de prueba:", semanas_test)

print("\nX_train:", X_train.shape)
print("X_test:", X_test.shape)

print("\nDistribución de entrenamiento:")
print(y_train.value_counts(normalize=True).round(3))

print("\nDistribución de prueba:")
print(y_test.value_counts(normalize=True).round(3))

Semanas disponibles: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16)]

Semanas de entrenamiento: [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)]
Semanas de prueba: [np.int64(13), np.int64(14), np.int64(15), np.int64(16)]

X_train: (3498, 13)
X_test: (288, 13)

Distribución de entrenamiento:
homeWin
1    0.572
0    0.428
Name: proportion, dtype: float64

Distribución de prueba:
homeWin
1    0.628
0    0.372
Name: proportion, dtype: float64


Para evitar que el codificador conozca categorías del conjunto de prueba durante su entrenamiento. Así simulamos mejor cómo funcionaría el modelo con partidos futuros.

Distribución de los datos
3,498
Entrenamiento · semanas 1–12
288
Prueba · semanas 13–16


Proporción de victorias locales
- Entrenamiento: 57.2%
- Prueba: 62.8%

Codificación de variables categóricas

Ahora sí vamos a transformar las variables categóricas utilizando OneHotEncoder.

Pero antes, una precisión: homeId y awayId tienen muchas categorías. Las vamos a codificar también, porque representan la identidad de los equipos. Utilizaremos matrices dispersas para controlar el consumo de memoria.

Definir las columnas categóricas y numéricas

In [18]:
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

# Variables categóricas
columnas_categoricas = [
    "seasonType",
    "homeClassification",
    "awayClassification",
    "homeConference",
    "awayConference",
    "homeId",
    "awayId"
]

# Variables numéricas y booleanas
columnas_numericas = [
    "week",
    "neutralSite",
    "conferenceGame",
    "startTimeTBD",
    "month",
    "dayOfWeek"
]

# Crear transformador
preprocesador = ColumnTransformer(
    transformers=[
        (
            "categoricas",
            OneHotEncoder(handle_unknown="ignore"),
            columnas_categoricas
        ),
        (
            "numericas",
            "passthrough",
            columnas_numericas
        )
    ]
)

# Ajustar únicamente con entrenamiento
X_train_encoded = preprocesador.fit_transform(X_train)

# Transformar prueba sin volver a ajustar
X_test_encoded = preprocesador.transform(X_test)

# Mostrar resultados
print("Entrenamiento codificado:", X_train_encoded.shape)
print("Prueba codificada:", X_test_encoded.shape)

print("\nTipo de matriz:", type(X_train_encoded))
print("Cantidad de características:", X_train_encoded.shape[1])

Entrenamiento codificado: (3498, 1545)
Prueba codificada: (288, 1545)

Tipo de matriz: <class 'scipy.sparse._csr.csr_matrix'>
Cantidad de características: 1545


¿Qué estamos consiguiendo?
- OneHotEncoder: convierte las categorías en columnas numéricas.
- handle_unknown="ignore": permite procesar equipos o categorías que no aparecieron durante el entrenamiento.
- passthrough: conserva las variables numéricas y booleanas.
- fit_transform: aprende las categorías usando exclusivamente entrenamiento.
- transform: aplica exactamente la misma transformación a prueba.

¿Qué significan estos resultados?

Resultado	Significado
(3498, 1545)	3,498 partidos para entrenar y 1,545 características
(288, 1545)	288 partidos para probar y las mismas 1,545 características
csr_matrix	Matriz dispersa que ahorra memoria
1545	Total de características después de One-Hot Encoding

Lo más importante: ambas matrices tienen exactamente 1,545 columnas. Esto significa que el codificador mantuvo la misma estructura entre entrenamiento y prueba.

verificar las características generadas

In [19]:
# Obtener nombres de las características generadas
nombres_features = preprocesador.get_feature_names_out()

print("Total de características:", len(nombres_features))

print("\nPrimeras 20 características:")
print(nombres_features[:20])

print("\n¿Está homeWin entre las características?")
print(any("homeWin" in nombre for nombre in nombres_features))

Total de características: 1545

Primeras 20 características:
['categoricas__seasonType_postseason' 'categoricas__seasonType_regular'
 'categoricas__homeClassification_Unknown'
 'categoricas__homeClassification_fbs'
 'categoricas__homeClassification_fcs'
 'categoricas__homeClassification_ii'
 'categoricas__homeClassification_iii'
 'categoricas__awayClassification_Unknown'
 'categoricas__awayClassification_fbs'
 'categoricas__awayClassification_fcs'
 'categoricas__awayClassification_ii'
 'categoricas__awayClassification_iii' 'categoricas__homeConference_ACC'
 'categoricas__homeConference_American Athletic'
 'categoricas__homeConference_American Rivers'
 'categoricas__homeConference_American Southwest'
 'categoricas__homeConference_Big 12'
 'categoricas__homeConference_Big Sky'
 'categoricas__homeConference_Big South-OVC'
 'categoricas__homeConference_Big Ten']

¿Está homeWin entre las características?
False


guardar el preprocesador

In [20]:
import joblib
import os

# Crear carpeta models si no existe
os.makedirs("../models", exist_ok=True)

# Guardar el preprocesador
joblib.dump(
    preprocesador,
    "../models/preprocesador.joblib"
)

print("Preprocesador guardado correctamente.")

Preprocesador guardado correctamente.


¿Por qué lo guardamos? Porque cuando nuestra aplicación reciba un partido nuevo, tendremos que transformar sus características utilizando el mismo codificador del entrenamiento. Así evitamos que cambie la estructura de las 1,545 características.

In [22]:
import joblib
import os

os.makedirs("../models", exist_ok=True)

datos_entrenamiento = {
    "X_train": X_train_encoded,
    "X_test": X_test_encoded,
    "y_train": y_train,
    "y_test": y_test
}

joblib.dump(
    datos_entrenamiento,
    "../models/datos_entrenamiento.joblib"
)

print("Datos de entrenamiento guardados correctamente.")

Datos de entrenamiento guardados correctamente.
